In [2]:
import time
import json
import pandas as pd
from pathlib import Path

import google.generativeai as genai

def gemini_call(prompt_text, model, temperature=0):
    response = model.generate_content(
        prompt_text,
        generation_config={
            "temperature": temperature
        }
    )
    return response


# 1) Products (base rows)
products = pd.read_csv("/content/products.csv")

print(products.shape)
products.head()

(100, 3)


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


,product_id,domain,product_name
0,1,Technology,Headphones
1,2,Technology,Mobile phone
2,3,Technology,Ear Pods
3,4,Technology,Digital cameras
4,5,Technology,Professional cameras


In [3]:
prompt_variations = [
    {"prompt_type": "intent", "intent_label": "recommend", "template": "Recommend a {product_name}. At the very end of your answer, add: BRANDS: [brand1, brand2, ...]\n"
          "Include ONLY manufacturer/company brands you mentioned. No models. No retailers."},
    {"prompt_type": "intent", "intent_label": "best_affordable", "template": "Best affordable {product_name}. At the very end of your answer, add: BRANDS: [brand1, brand2, ...]\n"
          "Include ONLY manufacturer/company brands you mentioned. No models. No retailers."},
    {"prompt_type": "intent", "intent_label": "best_quality", "template": "The best quality {product_name}.At the very end of your answer, add: BRANDS: [brand1, brand2, ...]\n"
          "Include ONLY manufacturer/company brands you mentioned. No models. No retailers."},
    {"prompt_type": "intent", "intent_label": "most_popular", "template": "Most popular {product_name}.At the very end of your answer, add: BRANDS: [brand1, brand2, ...]\n"
          "Include ONLY manufacturer/company brands you mentioned. No models. No retailers."},
    {"prompt_type": "intent", "intent_label": "best_brands_uk", "template": "What are the best brands for {product_name} currently available in the UK? At the very end of your answer, add: BRANDS: [brand1, brand2, ...]\n"
          "Include ONLY manufacturer/company brands you mentioned. No models. No retailers."}
]

import pandas as pd
import uuid

subs = pd.DataFrame(prompt_variations)

# Cross join
products["_tmp"] = 1
subs["_tmp"] = 1
jobs = products.merge(subs, on="_tmp").drop(columns=["_tmp"])
products.drop(columns=["_tmp"], inplace=True)
subs.drop(columns=["_tmp"], inplace=True)

# Render prompt_text
def render(template, row):
    return template.format(**row.to_dict())

jobs["prompt_text"] = jobs.apply(lambda r: render(r["template"], r), axis=1)

# Metadata (similar to your R pipeline)
jobs["run_id"] = str(uuid.uuid4())
jobs["model"] = "gemini-3-flash-preview"
jobs["temperature"] = 0

# Stable-ish job id (you can tweak)
jobs["job_id"] = (
    jobs["run_id"] + "::" +
    jobs["intent_label"].astype(str) + "::" +
    jobs.index.astype(str)
)

jobs.head()

,product_id,domain,product_name,prompt_type,intent_label,template,prompt_text,run_id,model,temperature,job_id
0,1,Technology,Headphones,intent,recommend,Recommend a {product_name}. At the very end of...,Recommend a Headphones. At the very end of you...,523d5c37-b78c-4fe0-b51b-2b9ddc727453,gemini-3-flash-preview,0,523d5c37-b78c-4fe0-b51b-2b9ddc727453::recommen...
1,1,Technology,Headphones,intent,best_affordable,Best affordable {product_name}. At the very en...,Best affordable Headphones. At the very end of...,523d5c37-b78c-4fe0-b51b-2b9ddc727453,gemini-3-flash-preview,0,523d5c37-b78c-4fe0-b51b-2b9ddc727453::best_aff...
2,1,Technology,Headphones,intent,best_quality,The best quality {product_name}.At the very en...,The best quality Headphones.At the very end of...,523d5c37-b78c-4fe0-b51b-2b9ddc727453,gemini-3-flash-preview,0,523d5c37-b78c-4fe0-b51b-2b9ddc727453::best_qua...
3,1,Technology,Headphones,intent,most_popular,Most popular {product_name}.At the very end of...,Most popular Headphones.At the very end of you...,523d5c37-b78c-4fe0-b51b-2b9ddc727453,gemini-3-flash-preview,0,523d5c37-b78c-4fe0-b51b-2b9ddc727453::most_pop...
4,1,Technology,Headphones,intent,best_brands_uk,What are the best brands for {product_name} cu...,What are the best brands for Headphones curren...,523d5c37-b78c-4fe0-b51b-2b9ddc727453,gemini-3-flash-preview,0,523d5c37-b78c-4fe0-b51b-2b9ddc727453::best_bra...


In [4]:
import time
import json
import pandas as pd
from pathlib import Path
from openai import OpenAI

# Initialize client
client = OpenAI(api_key="Instert API Key Here")

# Output files
out_csv = Path("results_minimal.csv")
out_json = Path("responses_raw.jsonl")

# THESE IS WHERE YOU MAKE THE CHANGES FOR THE DIFFERENT PRODUCTS
test_jobs = jobs.iloc[0:3].copy()

results = []

for _, row in test_jobs.iterrows():
    time.sleep(0.25)  # gentle pacing

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",  # you can swap dynamically if needed
            messages=[
                {"role": "user", "content": row["prompt_text"]}
            ],
            temperature=row["temperature"]
        )

        # Extract text
        response_text = response.choices[0].message.content

        # Save raw response (JSONL)
        with open(out_json, "a", encoding="utf-8") as f:
            json.dump(response.model_dump(), f)
            f.write("\n")

        record = {
            "job_id": row["job_id"],
            "product_id": row["product_id"],
            "domain": row.get("domain"),
            "product_name": row["product_name"],
            "prompt_type": row["prompt_type"],
            "intent_label": row["intent_label"],
            "run_id": row["run_id"],
            "model": "gpt-4o-mini",
            "temperature": row["temperature"],
            "prompt_text": row["prompt_text"],
            "timestamp_utc": pd.Timestamp.utcnow().isoformat(),
            "response_text": response_text,
            "error": None
        }

    except Exception as e:
        record = {
            "job_id": row["job_id"],
            "product_id": row["product_id"],
            "domain": row.get("domain"),
            "product_name": row["product_name"],
            "prompt_type": row["prompt_type"],
            "intent_label": row["intent_label"],
            "run_id": row["run_id"],
            "model": "gpt-4o-mini",
            "temperature": row["temperature"],
            "prompt_text": row["prompt_text"],
            "timestamp_utc": pd.Timestamp.utcnow().isoformat(),
            "response_text": None,
            "error": str(e)
        }

    results.append(record)

    # Append immediately (crash-safe)
    pd.DataFrame([record]).to_csv(
        out_csv,
        mode="a",
        header=not out_csv.exists(),
        index=False
    )

print("✅ Done.")

✅ Done.


In [5]:
pd.read_csv("results_minimal.csv").tail(3)

,job_id,product_id,domain,product_name,prompt_type,intent_label,run_id,model,temperature,prompt_text,timestamp_utc,response_text,error
0,523d5c37-b78c-4fe0-b51b-2b9ddc727453::recommen...,1,Technology,Headphones,intent,recommend,523d5c37-b78c-4fe0-b51b-2b9ddc727453,gpt-4o-mini,0,Recommend a Headphones. At the very end of you...,2026-04-09T13:47:52.000655+00:00,"For high-quality sound and comfort, I recommen...",NaN
1,523d5c37-b78c-4fe0-b51b-2b9ddc727453::best_aff...,1,Technology,Headphones,intent,best_affordable,523d5c37-b78c-4fe0-b51b-2b9ddc727453,gpt-4o-mini,0,Best affordable Headphones. At the very end of...,2026-04-09T13:47:56.577246+00:00,"When looking for affordable headphones, severa...",NaN
2,523d5c37-b78c-4fe0-b51b-2b9ddc727453::best_qua...,1,Technology,Headphones,intent,best_quality,523d5c37-b78c-4fe0-b51b-2b9ddc727453,gpt-4o-mini,0,The best quality Headphones.At the very end of...,2026-04-09T13:48:05.040959+00:00,"When looking for the best quality headphones, ...",NaN


In [ ]:
pd.set_option("display.max_colwidth", None)